# 00 · Data tour: VIX, PUT, DTB3

A hands-on look at the local data store. Run `python -m data.update` first (from `quant-research/`).

**Where the data lives** (outside the repo, `~/qr-data`, or `$QR_DATA_DIR`):

```
~/qr-data/
  raw/cboe/PUT/<YYYY-MM-DD>.csv   each day's download, exactly as received
  raw/cboe/VIX/<YYYY-MM-DD>.csv
  raw/fred/DTB3/<YYYY-MM-DD>.csv
  clean/indices.parquet           PUT + VIX: date, symbol, open, high, low, close
  clean/rates.parquet             DTB3:      date, symbol, value (% per year)
```

There is no database server or database file. The **clean Parquet files are the database**. DuckDB reads them on the fly: each file becomes a SQL table named after it (`indices`, `rates`).

In [ ]:
import sys
from pathlib import Path

# make `import data` work: find the quant-research folder above this notebook
QR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "store.py").exists())
sys.path.insert(0, str(QR))

from data import load, store

q = store.query   # shortcut: q("SELECT ...") returns a DataFrame
print("data store:", store.root())

## 1. What files are there?

In [ ]:
for f in sorted(store.root().rglob("*.*")):
    print(f"{f.stat().st_size / 1024:8.0f} KB  {f.relative_to(store.root())}")

In [ ]:
# peek at a raw file, exactly as Cboe sent it
raw = sorted((store.root() / "raw" / "cboe" / "VIX").glob("*.csv"))[-1]
print(raw)
print("".join(raw.read_text().splitlines(keepends=True)[:5]))

## 2. SQL queries on the clean tables

Edit and rerun freely. Tables: `indices` (PUT, VIX) and `rates` (DTB3).

In [ ]:
# coverage: first date, last date, row count per symbol
q("""
SELECT symbol, min(date) AS first, max(date) AS last, count(*) AS rows FROM indices GROUP BY symbol
UNION ALL
SELECT symbol, min(date), max(date), count(*) FROM rates GROUP BY symbol
""")

In [ ]:
# what a clean table looks like: the last 10 VIX days
q("SELECT * FROM indices WHERE symbol = 'VIX' ORDER BY date DESC LIMIT 10")

In [ ]:
# the 10 highest VIX closes ever (expect 2008 and March 2020)
q("SELECT date, close FROM indices WHERE symbol = 'VIX' ORDER BY close DESC LIMIT 10")

In [ ]:
# VIX per year: average, low, high
q("""
SELECT year(date) AS year,
       round(avg(close), 1) AS avg_vix,
       min(close)           AS low,
       max(close)           AS high
FROM indices WHERE symbol = 'VIX'
GROUP BY year ORDER BY year
""")

In [ ]:
# PUT: daily data only starts in 2007 (before that, a few scattered points)
q("SELECT year(date) AS year, count(*) AS days FROM indices WHERE symbol = 'PUT' GROUP BY year ORDER BY year")

In [ ]:
# PUT calendar-year returns: year-end close vs previous year-end close
q("""
WITH year_end AS (
    SELECT year(date) AS year, arg_max(close, date) AS close   -- close on the last date of the year
    FROM indices WHERE symbol = 'PUT' AND date >= '2007-01-01'
    GROUP BY year
)
SELECT year, close,
       round(100 * (close / lag(close) OVER (ORDER BY year) - 1), 1) AS return_pct
FROM year_end ORDER BY year
""")

In [ ]:
# T-bill rate (DTB3, % per year): monthly average, last 24 months
q("""
SELECT date_trunc('month', date) AS month, round(avg(value), 2) AS tbill_pct
FROM rates WHERE symbol = 'DTB3'
GROUP BY month ORDER BY month DESC LIMIT 24
""")

In [ ]:
# join across tables: VIX, PUT and the T-bill rate on the same day
q("""
SELECT v.date, v.close AS vix, p.close AS put, r.value AS tbill_pct
FROM indices v
JOIN indices p ON p.date = v.date AND p.symbol = 'PUT'
LEFT JOIN rates r ON r.date = v.date AND r.symbol = 'DTB3'
WHERE v.symbol = 'VIX'
ORDER BY v.date DESC LIMIT 10
""")

## 3. The pandas way: `load()`

This is how strategy code reads data: a panel with dates as rows and symbols as columns.

In [ ]:
px = load("indices", ["PUT", "VIX"], start="2007-01-01")
tbill = load("rates", ["DTB3"], field="value", start="2007-01-01")
px.tail()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
px["PUT"].plot(ax=axes[0], title="PUT (Cboe PutWrite index)")
px["VIX"].plot(ax=axes[1], title="VIX")
tbill["DTB3"].plot(ax=axes[2], title="3-month T-bill, % per year (FRED DTB3)")
plt.tight_layout()